# CCIA gm/ID Agent — Deployment Notebook

Loads a trained PPO checkpoint, rolls out the policy, plots all design-quality metrics,  
and saves every finalised design to CSV + JSON.


## 0 · Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'figure.dpi'       : 120,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})
%matplotlib inline

from pygmid import Lookup as lk
from ComputingParameters import CurrentReuse
from Deploy import DeploymentRunner

print('Imports OK')

## 1 · CONFIG  ← edit here

In [ ]:
# Filepaths 
MODEL_PATH    = 'runs/3rd_Run_1000_episodes/checkpoints/checkpoint_ep1000.pth'   
NCH_DATA_PATH = './XT018/LVT/180nch.mat'   
PCH_DATA_PATH = './XT018/LVT/180pch.mat'   
OUTPUT_DIR    = 'runs_deployment/3rd_Run_1000_episodes_Different_Specs_1000episode'  # Output directory 

# Run settings 
N_EPISODES  = 50   # number of deployment roll-outs
MAX_STEPS   = 100   # max steps per episode (match training)
SEED_OFFSET = 0     # change for different random starts

# Changing trained values
ENV_KWARGS = {
    'Fc_max'          : 40e3,
    # 'Cin'            : 10e-12,
    'vnin_target'    : 4.3,
    'area_budget_um2': 700,
}

print('Config OK')

## 2 · Load transistor data & build circuit model

In [ ]:
NCH = lk(NCH_DATA_PATH)
PCH = lk(PCH_DATA_PATH)

amp_functions = CurrentReuse(
    NCH = NCH,
    PCH = PCH,
    # Chaneg circuit parameters here
    # LP=1.5
)

print('Transistor data loaded and CurrentReuse instance ready.')

## 3 · Initialise the deployment runner

In [ ]:
runner = DeploymentRunner(
    model_path    = MODEL_PATH,
    amp_functions = amp_functions,
    output_dir    = OUTPUT_DIR,
    env_kwargs    = ENV_KWARGS,
    state_dim     = 8,    # must match trained model
    action_dim    = 4,    # must match trained model
    hidden_dim    = 128,  # must match trained model
)

## 4 · Run the agent

In [ ]:
designs = runner.run(
    n_episodes  = N_EPISODES,
    max_steps   = MAX_STEPS,
    seed_offset = SEED_OFFSET,
    verbose     = True,
)
print(f'\nGot {len(designs)} design records.')

## 5 · Save all designs

In [ ]:
csv_path, json_path = runner.save_designs(prefix='deployment')
print(f'CSV  : {csv_path}')
print(f'JSON : {json_path}')

## 6 · Quick summary table

In [ ]:
# Top 10 designs by VNin
top10 = df.nsmallest(10, 'vn_in_nV')[cols].reset_index(drop=True)
print('Top 10 designs by VNin (nV/√Hz)')
display(top10)